In [ ]:
import numpy as np
import os
import pandas as pd
import glob
import scipy.signal
from datetime import timedelta
from scipy.signal import butter, filtfilt
import matplotlib.pyplot as plt

# Combining Files to Save Space


In [ ]:
PROJECT_ROOT = "/Users/joemcdade/Desktop/MSci Project"
os.chdir(PROJECT_ROOT)

print("Working directory:", os.getcwd())
os.listdir()

In [ ]:
base_folder = "base"
extras_folder = "extras"
output_folder = "senior_data"

base_files = glob.glob(os.path.join(base_folder, "*-Team_on_pitch.csv"))

for base_path in base_files:
    filename = os.path.basename(base_path)

    extras_filename = filename.replace(
        "-Team_on_pitch.csv",
        "-Team_on_pitch_extras.csv"
    )
    extras_path = os.path.join(extras_folder, extras_filename)

    if not os.path.exists(extras_path):
        print(f"Missing extras file for {filename}")
        continue

    df1 = pd.read_csv(base_path)
    df2 = pd.read_csv(extras_path)

    combined = pd.concat([df1, df2.iloc[:, 2:]], axis=1)

    output_path = os.path.join(
        output_folder,
        filename.replace(
            "-Team_on_pitch.csv",
            "-Team_on_pitch_combined.csv"
        )
    )

    combined.to_csv(output_path, index=False)

print("Done.")


# Data Preprocessing

In [ ]:
df = pd.read_csv("") # enter file name here 

In [ ]:
imu_cols = [
    'Accl X', 'Accl Y', 'Accl Z',
    'Gyro Yro X', 'Gyro Y', 'Gyro Z'
]

meta_cols = [
    'Player Display Name', 'Time', 'Lat', 'Lon',
    'Speed (m/s)', 'Heart Rate (bpm)',
    'Hacc', 'Hdop', 'Quality of Signal',
    'No. of Satellites',
    'Instantaneous Acceleration Impulse'
]

def lowpass_filter(signal, fs=100, cutoff=4.0, order=4):
    b, a = butter(order, cutoff / (fs / 2), btype='low')
    return filtfilt(b, a, signal)

def resample_100hz_to_10hz(df):
    df = df.reset_index(drop=True)

    fs_old = 100
    fs_new = 10
    factor = fs_old // fs_new  # 10

    # Filtering IMU columns
    df_filtered = df.copy()
    for col in imu_cols:
        df_filtered[col] = lowpass_filter(df[col].values, fs=fs_old)

    # Downsampling IMU data
    imu_down = df_filtered[imu_cols].iloc[::factor].reset_index(drop=True)

    # Handling duplicated 10 Hz metadata 
    meta_down = (
        df[meta_cols]
        .groupby(df.index // factor)
        .first()        
        .reset_index(drop=True)
    )

    # Combining
    out = pd.concat([meta_down, imu_down], axis=1)

    return out

In [ ]:
df_10hz = resample_100hz_to_10hz(df)

In [ ]:
print(df.shape)      # original
print(df_10hz.shape) # should be 1/10th rows

In [ ]:
output_path = "" # enter output path here 
df_10hz.to_csv(output_path, index=False)

# Isolating Sprint Event

In [ ]:
df = pd.read_csv("") # enter file name here 

In [ ]:
list(df)

In [ ]:
import pandas as pd
import numpy as np
import scipy.signal
import matplotlib.pyplot as plt

def parse_time_column(time_column):
    try:
        return pd.to_datetime(time_column, format='%H:%M:%S.%f')
    except ValueError:
        return pd.to_datetime(time_column, format='%Y-%m-%d %H:%M:%S.%f')

def butter(speed):
    raw_speed = speed.values
    fs = FS
    nyquist = fs/2
    cutoff_freq = 2
    Wn = cutoff_freq / nyquist
    b, a = scipy.signal.butter(N=4, Wn=Wn, btype='low')
    filtered_speed = scipy.signal.filtfilt(b, a, raw_speed)
    return filtered_speed

In [ ]:
# Ensuring time is datetime
df['Time'] = parse_time_column(df['Time'])

# Sorting properly
df = df.sort_values(['Player Display Name', 'Time'])

In [ ]:
FS = 10 # sampling frequency
player_name = "" # enter player name here 

player_df = df[df['Player Display Name'] == player_name].copy()
player_df = player_df.sort_values('Time').reset_index(drop=True)

# Applying Butterworth filter
player_df['Filtered Speed'] = butter(player_df['Speed (m/s)'])

In [ ]:
SPRINT_THRESHOLD = 7  # m/s

player_df['Is Sprint'] = player_df['Filtered Speed'] > SPRINT_THRESHOLD

In [ ]:
# Identifying changes
player_df['Sprint Change'] = player_df['Is Sprint'].astype(int).diff()

# Sprinting starts
sprint_starts = player_df.index[player_df['Sprint Change'] == 1].tolist()

# Sprinting ends
sprint_ends = player_df.index[player_df['Sprint Change'] == -1].tolist()

# Handling edge cases
if len(sprint_ends) < len(sprint_starts):
    sprint_ends.append(player_df.index[-1])

In [ ]:
start_idx = sprint_starts[0]
end_idx = sprint_ends[0]

In [ ]:
JOG_THRESHOLD = 2

# Expanding backwards
i = start_idx
while i > 0 and player_df.loc[i, 'Filtered Speed'] > JOG_THRESHOLD:
    i -= 1
expanded_start = i

# Expanding forwards
i = end_idx
while i < len(player_df)-1 and player_df.loc[i, 'Filtered Speed'] > JOG_THRESHOLD:
    i += 1
expanded_end = i

In [ ]:
sprint_df = player_df.loc[expanded_start:expanded_end].copy()
sprint_df = sprint_df.reset_index(drop=True)

# Creating relative time (seconds)
sprint_df['Relative Time'] = (
    sprint_df['Time'] - sprint_df['Time'].iloc[0]
).dt.total_seconds()

In [ ]:
sprint_df.shape

In [ ]:
plt.figure()
plt.plot(sprint_df['Relative Time'], sprint_df['Filtered Speed'])
plt.xlabel("Time (s)")
plt.ylabel("Speed (m/s)")
plt.title("Sprint Event - Speed vs Time")
plt.savefig("isolated_sprint.png", dpi=300, bbox_inches='tight')
plt.show()

# Directional Acceleration 

In [ ]:
import numpy as np

# Earth radius
R = 6371000  # meters

# Reference point (start of sprint)
lat0 = np.deg2rad(sprint_df['Lat'].iloc[0])
lon0 = np.deg2rad(sprint_df['Lon'].iloc[0])

lat = np.deg2rad(sprint_df['Lat'].values)
lon = np.deg2rad(sprint_df['Lon'].values)

# Converting to local Cartesian coordinates
x = (lon - lon0) * np.cos(lat0) * R
y = (lat - lat0) * R

sprint_df['X'] = x
sprint_df['Y'] = y

In [ ]:
dt = 1 / FS

vx = np.gradient(sprint_df['X'], dt)
vy = np.gradient(sprint_df['Y'], dt)

sprint_df['Vx'] = vx
sprint_df['Vy'] = vy

In [ ]:
heading = np.arctan2(vy, vx)
sprint_df['Heading'] = heading

# Converting to degrees for readability
sprint_df['Heading_deg'] = np.degrees(heading)

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(sprint_df['X'], sprint_df['Y'])

# Adding arrow showing overall sprint direction
plt.quiver(
    sprint_df['X'].iloc[0],
    sprint_df['Y'].iloc[0],
    sprint_df['X'].iloc[-1] - sprint_df['X'].iloc[0],
    sprint_df['Y'].iloc[-1] - sprint_df['Y'].iloc[0],
    angles='xy',
    scale_units='xy',
    scale=1
)

plt.xlabel("X Position (m)")
plt.ylabel("Y Position (m)")
plt.title("Sprint Direction (Arrow Representation)")
plt.axis('equal')
plt.savefig("isolated_sprint_direction.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import scipy.signal
import matplotlib.pyplot as plt

# Setting Parameters 
SPRINT_THRESHOLD = 7.0   # m/s (sprint definition)
JOG_THRESHOLD    = 2.0   # m/s (to expand start/end to "jog")
MIN_SPRINT_DUR_S = 0.8   # seconds (discard tiny false positives)
R = 6371000              # Earth radius (m)

player_name = "" # enter player name here 

# Player slice 
player_df = df[df['Player Display Name'] == player_name].copy()
player_df['Time'] = parse_time_column(player_df['Time'])
player_df = player_df.sort_values('Time').reset_index(drop=True)

# Filtered speed 
player_df['Filtered Speed'] = butter(player_df['Speed (m/s)'])

# Local XY coordinates (meters), common reference for the match 
lat0 = np.deg2rad(player_df['Lat'].iloc[0])
lon0 = np.deg2rad(player_df['Lon'].iloc[0])
lat = np.deg2rad(player_df['Lat'].to_numpy())
lon = np.deg2rad(player_df['Lon'].to_numpy())

player_df['X'] = (lon - lon0) * np.cos(lat0) * R
player_df['Y'] = (lat - lat0) * R

In [ ]:
def find_sprints(player_df, FS, sprint_thr=7.0, jog_thr=2.0, min_dur_s=0.8):
    is_sprint = (player_df['Filtered Speed'] > sprint_thr).to_numpy().astype(int)
    change = np.diff(is_sprint, prepend=is_sprint[0])

    starts = np.where(change == 1)[0].tolist()
    ends   = np.where(change == -1)[0].tolist()

    # Edge cases
    if is_sprint[0] == 1 and (len(starts) == 0 or starts[0] != 0):
        starts = [0] + starts
    if len(ends) < len(starts):
        ends.append(len(player_df) - 1)

    expanded = []
    min_len = int(np.ceil(min_dur_s * FS))

    for s, e in zip(starts, ends):
        # Expanding backward until <= jog_thr
        i = s
        while i > 0 and player_df.loc[i, 'Filtered Speed'] > jog_thr:
            i -= 1
        s_exp = i

        # Expanding forward until <= jog_thr
        i = e
        while i < len(player_df) - 1 and player_df.loc[i, 'Filtered Speed'] > jog_thr:
            i += 1
        e_exp = i

        if (e_exp - s_exp + 1) >= min_len:
            expanded.append((s_exp, e_exp))

    # Merging overlaps (can happen if thresholds cause expansions to touch)
    merged = []
    for s, e in sorted(expanded):
        if not merged or s > merged[-1][1]:
            merged.append([s, e])
        else:
            merged[-1][1] = max(merged[-1][1], e)

    return [(s, e) for s, e in merged]

sprint_windows = find_sprints(
    player_df, FS,
    sprint_thr=SPRINT_THRESHOLD,
    jog_thr=JOG_THRESHOLD,
    min_dur_s=MIN_SPRINT_DUR_S
)

len(sprint_windows), sprint_windows[:5]

In [ ]:
plt.figure()

for k, (s_idx, e_idx) in enumerate(sprint_windows, start=1):
    seg = player_df.loc[s_idx:e_idx]

    # Path
    plt.plot(seg['X'], seg['Y'])

    # Arrow: start -> end displacement (overall direction)
    dx = seg['X'].iloc[-1] - seg['X'].iloc[0]
    dy = seg['Y'].iloc[-1] - seg['Y'].iloc[0]
    plt.quiver(
        seg['X'].iloc[0], seg['Y'].iloc[0],
        dx, dy,
        angles='xy', scale_units='xy', scale=1
    )

plt.xlabel("X Position (m)")
plt.ylabel("Y Position (m)")
plt.title(f"All Sprints for the player (Trajectories + Direction Arrows)")
plt.axis('equal')
plt.savefig("all_sprints_arrows.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import scipy.signal
import matplotlib.pyplot as plt

imu_df = pd.read_csv("") # enter file name here

# Rename this if your 100 Hz dataframe has a different name
imu_player_df = imu_df[imu_df['Player Display Name'] == player_name].copy()

imu_player_df['Time'] = parse_time_column(imu_player_df['Time'])
imu_player_df = imu_player_df.sort_values('Time').reset_index(drop=True)

imu_scale = 1.0

imu_player_df['Accl X'] = imu_player_df['Accl X'] * imu_scale
imu_player_df['Accl Y'] = imu_player_df['Accl Y'] * imu_scale

In [ ]:
def estimate_fs_from_time(time_series):
    dt = pd.Series(time_series).diff().dt.total_seconds().dropna()
    return 1.0 / dt.median()

def butter_lowpass_array(x, fs, cutoff=5.0, order=4):
    x = np.asarray(x, dtype=float)
    if len(x) < 20:
        return x.copy()
    
    nyquist = fs / 2.0
    Wn = cutoff / nyquist
    b, a = scipy.signal.butter(order, Wn, btype='low')
    
    # filtfilt needs enough samples
    padlen = 3 * (max(len(a), len(b)) - 1)
    if len(x) <= padlen:
        return x.copy()
    
    return scipy.signal.filtfilt(b, a, x)

def get_noise_band(ax_raw, ay_raw, t_imu, edge_seconds=0.30):
    """
    Estimate a noise band from the first/last edge_seconds of the sprint window.
    Returns lower and upper bounds for a shaded band.
    """
    tmax = t_imu.max()
    edge_mask = (t_imu <= edge_seconds) | (t_imu >= (tmax - edge_seconds))
    
    # Fallback if sprint is very short
    if edge_mask.sum() < 20:
        n = len(t_imu)
        k = max(5, int(0.1 * n))
        edge_mask = np.zeros(n, dtype=bool)
        edge_mask[:k] = True
        edge_mask[-k:] = True
    
    # Baseline-center using edge medians
    ax_centered = ax_raw - np.median(ax_raw[edge_mask])
    ay_centered = ay_raw - np.median(ay_raw[edge_mask])
    
    baseline_vals = np.r_[ax_centered[edge_mask], ay_centered[edge_mask]]
    low, high = np.quantile(baseline_vals, [0.05, 0.95])
    
    return low, high, ax_centered, ay_centered

In [ ]:
FS_IMU = 100

print(f"Number of sprint windows: {len(sprint_windows)}")

In [ ]:
for sprint_num, (s_idx, e_idx) in enumerate(sprint_windows, start=1):
    # -----------------------------
    # GPS segment (10 Hz)
    # -----------------------------
    gps_seg = player_df.loc[s_idx:e_idx].copy().reset_index(drop=True)
    
    t0 = gps_seg['Time'].iloc[0]
    t1 = gps_seg['Time'].iloc[-1]
    
    t_gps = (gps_seg['Time'] - t0).dt.total_seconds().to_numpy()
    speed_gps = gps_seg['Filtered Speed'].to_numpy()
    
    # Derivative of filtered GPS speed
    gps_acc = np.gradient(speed_gps, t_gps)

    # -----------------------------
    # IMU segment (100 Hz)
    # -----------------------------
    imu_seg = imu_player_df[
        (imu_player_df['Time'] >= t0) & (imu_player_df['Time'] <= t1)
    ].copy().reset_index(drop=True)
    
    # Skip if no IMU data found
    if len(imu_seg) == 0:
        print(f"Sprint {sprint_num}: no matching IMU samples found.")
        continue
    
    t_imu = (imu_seg['Time'] - t0).dt.total_seconds().to_numpy()
    ax_raw = imu_seg['Accl X'].to_numpy()
    ay_raw = imu_seg['Accl Y'].to_numpy()
    
    # Noise band + baseline centering
    noise_low, noise_high, ax_centered, ay_centered = get_noise_band(
        ax_raw, ay_raw, t_imu, edge_seconds=0.30
    )
    
    # Smooth the IMU traces
    ax_smooth = butter_lowpass_array(ax_centered, FS_IMU, cutoff=5.0, order=4)
    ay_smooth = butter_lowpass_array(ay_centered, FS_IMU, cutoff=5.0, order=4)

    # -----------------------------
    # Plot
    # -----------------------------
    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(11, 7), sharex=True, constrained_layout=True
    )
    
    # Top panel: speed
    ax1.plot(t_gps, speed_gps, label='Filtered GPS speed')
    ax1.set_ylabel('Speed (m/s)')
    ax1.set_title(f'Sprint {sprint_num}: Speed and Acceleration Comparison')
    ax1.legend(loc='best')
    ax1.grid(True, alpha=0.3)
    
    # Bottom panel: acceleration comparison
    ax2.axhspan(noise_low, noise_high, alpha=0.2, label='IMU edge noise band')
    
    ax2.plot(t_imu, ax_centered, alpha=0.35, label='a_x raw (100 Hz)')
    ax2.plot(t_imu, ay_centered, alpha=0.35, label='a_y raw (100 Hz)')
    
    ax2.plot(t_imu, ax_smooth, linewidth=2, label='a_x smoothed')
    ax2.plot(t_imu, ay_smooth, linewidth=2, label='a_y smoothed')
    
    ax2.plot(t_gps, gps_acc, linewidth=2, label='d(speed)/dt from GPS (10 Hz)')
    
    ax2.set_xlabel('Time from sprint start (s)')
    ax2.set_ylabel('Acceleration')
    ax2.legend(loc='best', ncol=2)
    ax2.grid(True, alpha=0.3)
    
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
import scipy.signal
from pathlib import Path

calib_files = {
    "forward.csv": "forward",
    "back.csv": "back",
    "left.csv": "left",
    "right.csv": "right",
    "up.csv": "up",
    "down.csv": "down",
    "clockwise.csv": "clockwise",
    "anticlockwise.csv": "anticlockwise",
}

calib_list = []

for file, label in calib_files.items():
    temp = pd.read_csv(file)
    temp['label'] = label
    temp['Time'] = parse_time_column(temp['Time'])
    calib_list.append(temp)

calib_df = pd.concat(calib_list, ignore_index=True)

In [ ]:
def lowpass_array(x, fs, cutoff=5, order=4):
    x = np.asarray(x, dtype=float)
    nyquist = fs / 2
    Wn = cutoff / nyquist
    b, a = scipy.signal.butter(order, Wn, btype='low')
    return scipy.signal.filtfilt(b, a, x)

FS_IMU = 100  # if known

for col in ['Accl X', 'Accl Y', 'Accl Z', 'Gyro Yro X', 'Gyro Y', 'Gyro Z']:
    calib_df[f'{col}_filt'] = lowpass_array(calib_df[col], FS_IMU, cutoff=5)

In [ ]:
summary = calib_df.groupby('label')[
    ['Accl X_filt', 'Accl Y_filt', 'Accl Z_filt',
     'Gyro Yro X_filt', 'Gyro Y_filt', 'Gyro Z_filt']
].median()

summary

In [ ]:
import numpy as np
import pandas as pd
import scipy.signal

FS_IMU = 100

def lowpass_array(x, fs, cutoff=10, order=4):
    x = np.asarray(x, dtype=float)
    nyquist = fs / 2
    Wn = cutoff / nyquist
    b, a = scipy.signal.butter(order, Wn, btype='low')
    padlen = 3 * (max(len(a), len(b)) - 1)
    if len(x) <= padlen:
        return x.copy()
    return scipy.signal.filtfilt(b, a, x)

def summarise_trial_dynamic(df_trial, fs=100):
    out = {}
    
    for col in ['Accl X', 'Accl Y', 'Accl Z', 'Gyro Yro X', 'Gyro Y', 'Gyro Z']:
        x = df_trial[col].to_numpy(dtype=float)
        x = lowpass_array(x, fs, cutoff=10, order=4)
        
        n = len(x)
        edge = max(5, int(0.15 * n))
        mid0 = int(0.25 * n)
        mid1 = int(0.75 * n)
        
        # baseline from start/end
        baseline = np.median(np.r_[x[:edge], x[-edge:]])
        x_centered = x - baseline
        
        x_mid = x_centered[mid0:mid1]
        
        out[f'{col}_mean_mid'] = np.mean(x_mid)
        out[f'{col}_peak_pos'] = np.max(x_mid)
        out[f'{col}_peak_neg'] = np.min(x_mid)
        out[f'{col}_peak_abs'] = np.max(np.abs(x_mid))
        out[f'{col}_impulse'] = np.trapz(x_mid, dx=1/fs)
    
    return pd.Series(out)

In [ ]:
dynamic_summary = calib_df.groupby('label').apply(summarise_trial_dynamic, fs=FS_IMU)
dynamic_summary.T

In [ ]:
imu_player_df['a_forward'] = imu_player_df['Accl Z']
imu_player_df['a_lateral'] = imu_player_df['Accl X']
imu_player_df['a_vertical'] = -imu_player_df['Accl Y']
imu_player_df['yaw_rate'] = imu_player_df['Gyro Y']

In [ ]:
def butter_lowpass_array(x, fs, cutoff=5.0, order=4):
    x = np.asarray(x, dtype=float)

    if len(x) < 20:
        return x.copy()

    nyquist = fs / 2.0
    Wn = cutoff / nyquist
    b, a = scipy.signal.butter(order, Wn, btype='low')

    padlen = 3 * (max(len(a), len(b)) - 1)
    if len(x) <= padlen:
        return x.copy()

    return scipy.signal.filtfilt(b, a, x)

def baseline_center_from_edges(x, edge_frac=0.15):
    x = np.asarray(x, dtype=float)
    n = len(x)
    edge_n = max(5, int(edge_frac * n))
    baseline = np.median(np.r_[x[:edge_n], x[-edge_n:]])
    x_centered = x - baseline
    return x_centered, baseline, edge_n

def noise_band_from_edges(x_centered, edge_n):
    edge_vals = np.r_[x_centered[:edge_n], x_centered[-edge_n:]]
    low, high = np.quantile(edge_vals, [0.05, 0.95])
    return low, high

In [ ]:
for sprint_num, (s_idx, e_idx) in enumerate(sprint_windows, start=1):
    gps_seg = player_df.loc[s_idx:e_idx].copy().reset_index(drop=True)

    t0 = gps_seg['Time'].iloc[0]
    t1 = gps_seg['Time'].iloc[-1]

    t_gps = (gps_seg['Time'] - t0).dt.total_seconds().to_numpy()
    speed_gps = gps_seg['Filtered Speed'].to_numpy()

    if len(t_gps) > 1:
        gps_acc = np.gradient(speed_gps, t_gps)
    else:
        gps_acc = np.zeros_like(speed_gps)

    imu_seg = imu_player_df[
        (imu_player_df['Time'] >= t0) & (imu_player_df['Time'] <= t1)
    ].copy().reset_index(drop=True)

    if len(imu_seg) == 0:
        print(f"Sprint {sprint_num}: no matching IMU data")
        continue

    t_imu = (imu_seg['Time'] - t0).dt.total_seconds().to_numpy()

    a_forward_raw = imu_seg['a_forward'].to_numpy()
    a_lateral_raw = imu_seg['a_lateral'].to_numpy()

    a_forward_centered, _, edge_n_fwd = baseline_center_from_edges(
        a_forward_raw, edge_frac=0.15
    )
    a_lateral_centered, _, edge_n_lat = baseline_center_from_edges(
        a_lateral_raw, edge_frac=0.15
    )

    a_forward_smooth = butter_lowpass_array(
        a_forward_centered, fs=FS_IMU, cutoff=5.0, order=4
    )
    a_lateral_smooth = butter_lowpass_array(
        a_lateral_centered, fs=FS_IMU, cutoff=5.0, order=4
    )

    noise_low, noise_high = noise_band_from_edges(a_forward_centered, edge_n_fwd)

    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(11, 7), sharex=True, constrained_layout=True
    )

    ax1.plot(t_gps, speed_gps, label='Filtered GPS speed')
    ax1.set_ylabel('Speed (m/s)')
    ax1.set_title(f'Sprint {sprint_num}: Speed and IMU Acceleration')
    ax1.legend(loc='best')
    ax1.grid(True, alpha=0.3)

    ax2.axhspan(noise_low, noise_high, alpha=0.2, label='IMU edge noise band')
    ax2.plot(t_imu, a_forward_centered, alpha=0.25, label='a_forward raw (100 Hz)')
    ax2.plot(t_imu, a_lateral_centered, alpha=0.25, label='a_lateral raw (100 Hz)')
    ax2.plot(t_imu, a_forward_smooth, linewidth=2, label='a_forward smoothed')
    ax2.plot(t_imu, a_lateral_smooth, linewidth=2, label='a_lateral smoothed')
    ax2.plot(t_gps, gps_acc, linewidth=2, label='d(speed)/dt from GPS (10 Hz)')

    ax2.set_xlabel('Time from sprint start (s)')
    ax2.set_ylabel('Acceleration')
    ax2.legend(loc='best', ncol=2)
    ax2.grid(True, alpha=0.3)

    plt.show()

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages

with PdfPages("all_sprints.pdf") as pdf:
    for sprint_num, (s_idx, e_idx) in enumerate(sprint_windows, start=1):
        # ----------------------------
        # GPS segment
        # ----------------------------
        gps_seg = player_df.loc[s_idx:e_idx].copy().reset_index(drop=True)

        t0 = gps_seg['Time'].iloc[0]
        t1 = gps_seg['Time'].iloc[-1]

        t_gps = (gps_seg['Time'] - t0).dt.total_seconds().to_numpy()
        speed_gps = gps_seg['Filtered Speed'].to_numpy()

        if len(t_gps) > 1:
            gps_acc = np.gradient(speed_gps, t_gps)
        else:
            gps_acc = np.zeros_like(speed_gps)

        # ----------------------------
        # IMU segment
        # ----------------------------
        imu_seg = imu_player_df[
            (imu_player_df['Time'] >= t0) & (imu_player_df['Time'] <= t1)
        ].copy().reset_index(drop=True)

        if len(imu_seg) == 0:
            print(f"Sprint {sprint_num}: no matching IMU data")
            continue

        t_imu = (imu_seg['Time'] - t0).dt.total_seconds().to_numpy()

        a_forward_raw = imu_seg['a_forward'].to_numpy()
        a_lateral_raw = imu_seg['a_lateral'].to_numpy()

        a_forward_centered, _, edge_n_fwd = baseline_center_from_edges(
            a_forward_raw, edge_frac=0.15
        )
        a_lateral_centered, _, _ = baseline_center_from_edges(
            a_lateral_raw, edge_frac=0.15
        )

        a_forward_smooth = butter_lowpass_array(
            a_forward_centered, fs=FS_IMU, cutoff=5.0, order=4
        )
        a_lateral_smooth = butter_lowpass_array(
            a_lateral_centered, fs=FS_IMU, cutoff=5.0, order=4
        )

        noise_low, noise_high = noise_band_from_edges(a_forward_centered, edge_n_fwd)

        # ----------------------------
        # Plot
        # ----------------------------
        fig, (ax1, ax2) = plt.subplots(
            2, 1, figsize=(11, 7), sharex=True, constrained_layout=True
        )

        ax1.plot(t_gps, speed_gps, label='Filtered GPS speed')
        ax1.set_ylabel('Speed (m/s)')
        ax1.set_title(f'Sprint {sprint_num}: Speed and IMU Acceleration')
        ax1.legend(loc='best')
        ax1.grid(True, alpha=0.3)

        ax2.axhspan(noise_low, noise_high, alpha=0.2, label='IMU edge noise band')
        ax2.plot(t_imu, a_forward_centered, alpha=0.25, label='a_forward raw (100 Hz)')
        ax2.plot(t_imu, a_lateral_centered, alpha=0.25, label='a_lateral raw (100 Hz)')
        ax2.plot(t_imu, a_forward_smooth, linewidth=2, label='a_forward smoothed')
        ax2.plot(t_imu, a_lateral_smooth, linewidth=2, label='a_lateral smoothed')
        ax2.plot(t_gps, gps_acc, linewidth=2, label='d(speed)/dt from GPS (10 Hz)')

        ax2.set_xlabel('Time from sprint start (s)')
        ax2.set_ylabel('Acceleration')
        ax2.legend(loc='best', ncol=2)
        ax2.grid(True, alpha=0.3)

        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

print("Saved as all_sprints.pdf")

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
import matplotlib.pyplot as plt

with PdfPages("all_sprints_masked_gps_acc.pdf") as pdf:
    for sprint_num, (s_idx, e_idx) in enumerate(sprint_windows, start=1):
        # ----------------------------
        # GPS segment
        # ----------------------------
        gps_seg = player_df.loc[s_idx:e_idx].copy().reset_index(drop=True)

        t0 = gps_seg['Time'].iloc[0]
        t1 = gps_seg['Time'].iloc[-1]

        t_gps = (gps_seg['Time'] - t0).dt.total_seconds().to_numpy()
        speed_gps = gps_seg['Filtered Speed'].to_numpy()

        if len(t_gps) > 1:
            gps_acc = np.gradient(speed_gps, t_gps)
        else:
            gps_acc = np.zeros_like(speed_gps)

        # ----------------------------
        # IMU segment
        # ----------------------------
        imu_seg = imu_player_df[
            (imu_player_df['Time'] >= t0) & (imu_player_df['Time'] <= t1)
        ].copy().reset_index(drop=True)

        if len(imu_seg) == 0:
            print(f"Sprint {sprint_num}: no matching IMU data")
            continue

        t_imu = (imu_seg['Time'] - t0).dt.total_seconds().to_numpy()

        a_forward_raw = imu_seg['a_forward'].to_numpy()
        a_lateral_raw = imu_seg['a_lateral'].to_numpy()

        # Baseline-center both
        a_forward_centered, _, edge_n_fwd = baseline_center_from_edges(
            a_forward_raw, edge_frac=0.15
        )
        a_lateral_centered, _, _ = baseline_center_from_edges(
            a_lateral_raw, edge_frac=0.15
        )

        # Smooth both
        a_forward_smooth = butter_lowpass_array(
            a_forward_centered, fs=FS_IMU, cutoff=5.0, order=4
        )
        a_lateral_smooth = butter_lowpass_array(
            a_lateral_centered, fs=FS_IMU, cutoff=5.0, order=4
        )

        # ----------------------------
        # IMU noise band -> GPS mask
        # ----------------------------
        noise_low, noise_high = noise_band_from_edges(a_forward_centered, edge_n_fwd)

        # Symmetric threshold based on the larger magnitude side
        noise_threshold = max(abs(noise_low), abs(noise_high))

        # Keep only GPS accelerations outside the IMU noise threshold
        gps_acc_masked = np.where(np.abs(gps_acc) > noise_threshold, gps_acc, np.nan)

        # Optional stricter version:
        # noise_threshold = 1.25 * max(abs(noise_low), abs(noise_high))
        # gps_acc_masked = np.where(np.abs(gps_acc) > noise_threshold, gps_acc, np.nan)

        # ----------------------------
        # Plot
        # ----------------------------
        fig, (ax1, ax2) = plt.subplots(
            2, 1, figsize=(11, 7), sharex=True, constrained_layout=True
        )

        # Top panel: speed
        ax1.plot(t_gps, speed_gps, label='Filtered GPS speed')
        ax1.set_ylabel('Speed (m/s)')
        ax1.set_title(f'Sprint {sprint_num}: Speed and Thresholded GPS Acceleration')
        ax1.legend(loc='best')
        ax1.grid(True, alpha=0.3)

        # Bottom panel: IMU + GPS acceleration comparison
        ax2.axhspan(noise_low, noise_high, alpha=0.2, label='IMU edge noise band')

        ax2.plot(t_imu, a_forward_centered, alpha=0.20, label='a_forward raw (100 Hz)')
        ax2.plot(t_imu, a_lateral_centered, alpha=0.20, label='a_lateral raw (100 Hz)')

        ax2.plot(t_imu, a_forward_smooth, linewidth=2, label='a_forward smoothed')
        ax2.plot(t_imu, a_lateral_smooth, linewidth=2, label='a_lateral smoothed')

        ax2.plot(t_gps, gps_acc, alpha=0.35, linewidth=1.5, label='GPS d(speed)/dt raw')
        ax2.plot(
            t_gps,
            gps_acc_masked,
            linewidth=2.5,
            label='GPS d(speed)/dt outside IMU noise band'
        )

        ax2.set_xlabel('Time from sprint start (s)')
        ax2.set_ylabel('Acceleration')
        ax2.legend(loc='best', ncol=2)
        ax2.grid(True, alpha=0.3)

        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

print("Saved as all_sprints_masked_gps_acc.pdf")

# Direction using GPS derivatives 

In [ ]:
import numpy as np
import pandas as pd
import scipy.signal
import matplotlib.pyplot as plt

In [ ]:
def butter_lowpass_array(x, fs, cutoff=2.0, order=4):
    x = np.asarray(x, dtype=float)

    if len(x) < 20:
        return x.copy()

    nyquist = fs / 2.0
    Wn = cutoff / nyquist
    b, a = scipy.signal.butter(order, Wn, btype='low')

    padlen = 3 * (max(len(a), len(b)) - 1)
    if len(x) <= padlen:
        return x.copy()

    return scipy.signal.filtfilt(b, a, x)

def wrap_to_pi(angle):
    return (angle + np.pi) % (2 * np.pi) - np.pi

def add_local_xy(df_in):
    R = 6371000  # m
    out = df_in.copy()

    lat0 = np.deg2rad(out['Lat'].iloc[0])
    lon0 = np.deg2rad(out['Lon'].iloc[0])

    lat = np.deg2rad(out['Lat'].to_numpy())
    lon = np.deg2rad(out['Lon'].to_numpy())

    out['X'] = (lon - lon0) * np.cos(lat0) * R
    out['Y'] = (lat - lat0) * R
    return out

In [ ]:
# Make sure player_df is clean and has filtered speed
player_df = player_df.copy()
player_df = player_df.dropna(subset=['Time']).sort_values('Time').drop_duplicates(subset=['Time']).reset_index(drop=True)

if 'Filtered Speed' not in player_df.columns:
    player_df['Filtered Speed'] = butter(player_df['Speed (m/s)'])

if 'X' not in player_df.columns or 'Y' not in player_df.columns:
    player_df = add_local_xy(player_df)

# Smooth XY as well, so heading changes are less noisy
player_df['X_filt'] = butter_lowpass_array(player_df['X'], fs=FS, cutoff=2.0, order=4)
player_df['Y_filt'] = butter_lowpass_array(player_df['Y'], fs=FS, cutoff=2.0, order=4)

In [ ]:
def build_accel_direction_cloud(player_df, sprint_windows, min_speed_for_direction=2.0):
    records = []

    for sprint_num, (s_idx, e_idx) in enumerate(sprint_windows, start=1):
        seg = player_df.loc[s_idx:e_idx].copy().reset_index(drop=True)

        # Need at least 4 samples to form central interval-based points
        if len(seg) < 4:
            continue

        t = (seg['Time'] - seg['Time'].iloc[0]).dt.total_seconds().to_numpy(dtype=float)
        x = seg['X_filt'].to_numpy(dtype=float)
        y = seg['Y_filt'].to_numpy(dtype=float)
        s = seg['Filtered Speed'].to_numpy(dtype=float)

        dt = np.diff(t)
        if np.any(dt <= 0):
            continue

        dx = np.diff(x)
        dy = np.diff(y)

        # Heading of each interval t[i] -> t[i+1]
        heading = np.unwrap(np.arctan2(dy, dx))  # length n-1

        # Front/back acceleration on each interval
        a_fb_interval = np.diff(s) / dt  # length n-1

        # Turn rate and lateral acceleration at the central samples
        dtheta = wrap_to_pi(np.diff(heading))           # length n-2
        dt_c = 0.5 * (dt[:-1] + dt[1:])                # length n-2
        speed_c = s[1:-1]                              # length n-2
        a_fb_c = 0.5 * (a_fb_interval[:-1] + a_fb_interval[1:])  # length n-2

        # Positive = left turn, negative = right turn
        a_lat = speed_c * (dtheta / dt_c)              # m/s^2

        t_c = t[1:-1]
        heading_deg = np.degrees(heading[1:])

        valid = (
            np.isfinite(t_c)
            & np.isfinite(a_fb_c)
            & np.isfinite(a_lat)
            & np.isfinite(speed_c)
            & (speed_c >= min_speed_for_direction)
        )

        for ti, spd, afb, alat, hdg in zip(
            t_c[valid], speed_c[valid], a_fb_c[valid], a_lat[valid], heading_deg[valid]
        ):
            # For plotting, flip sign so LEFT appears on the left side of the plot
            lr_plot = -alat

            if alat > 0:
                turn_dir = 'left'
            elif alat < 0:
                turn_dir = 'right'
            else:
                turn_dir = 'straight'

            records.append({
                'sprint_num': sprint_num,
                'time_s': ti,
                'speed_m_s': spd,
                'a_fb': afb,                 # + accelerating, - decelerating
                'a_lat_leftpos': alat,       # + left, - right
                'a_lr_plot': lr_plot,        # - left, + right (for intuitive plotting)
                'heading_deg': hdg,
                'turn_dir': turn_dir
            })

    points_df = pd.DataFrame.from_records(records)

    if len(points_df) == 0:
        return points_df

    points_df['quadrant'] = np.select(
        [
            (points_df['a_lr_plot'] < 0) & (points_df['a_fb'] > 0),
            (points_df['a_lr_plot'] > 0) & (points_df['a_fb'] > 0),
            (points_df['a_lr_plot'] < 0) & (points_df['a_fb'] < 0),
            (points_df['a_lr_plot'] > 0) & (points_df['a_fb'] < 0),
        ],
        [
            'accelerating_forward_left',
            'accelerating_forward_right',
            'decelerating_left',
            'decelerating_right',
        ],
        default='axis_or_zero'
    )

    return points_df

In [ ]:
points_df = build_accel_direction_cloud(
    player_df,
    sprint_windows,
    min_speed_for_direction=2.0
)

print(f"Total datapoints: {len(points_df)}")
print(f"Number of sprints represented: {points_df['sprint_num'].nunique()}")
points_df.head()

In [ ]:
def plot_topdown_accel_cloud(points_df):
    fig, ax = plt.subplots(figsize=(9, 9))

    ax.scatter(points_df['a_lr_plot'], points_df['a_fb'], alpha=0.2, s=18)

    ax.axhline(0)
    ax.axvline(0)

    # Symmetric limits from the 99th percentile
    x_lim = np.nanquantile(np.abs(points_df['a_lr_plot']), 0.99)
    y_lim = np.nanquantile(np.abs(points_df['a_fb']), 0.99)

    x_lim = max(0.5, x_lim) * 1.1
    y_lim = max(0.5, y_lim) * 1.1

    ax.set_xlim(-x_lim, x_lim)
    ax.set_ylim(-y_lim, y_lim)

    ax.set_xlabel('Left / Right acceleration (m/s²)\nLeft ←        → Right')
    ax.set_ylabel('Front / Back acceleration (m/s²)\nDecelerating ↓        ↑ Accelerating')
    ax.set_title(f'Top-down acceleration map across {points_df["sprint_num"].nunique()} sprints')

    ax.text(-0.60 * x_lim,  0.82 * y_lim, 'Accelerating\nforward-left', ha='center', va='center')
    ax.text( 0.60 * x_lim,  0.82 * y_lim, 'Accelerating\nforward-right', ha='center', va='center')
    ax.text(-0.60 * x_lim, -0.82 * y_lim, 'Decelerating\nleft', ha='center', va='center')
    ax.text( 0.60 * x_lim, -0.82 * y_lim, 'Decelerating\nright', ha='center', va='center')

    ax.grid(True, alpha=0.3)
    plt.show()

plot_topdown_accel_cloud(points_df)

In [ ]:
quad_counts = points_df[points_df['quadrant'] != 'axis_or_zero']['quadrant'].value_counts()
quad_counts

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))

ax.hexbin(points_df['a_lr_plot'], points_df['a_fb'], gridsize=35, mincnt=1)
ax.axhline(0)
ax.axvline(0)

x_lim = np.nanquantile(np.abs(points_df['a_lr_plot']), 0.99)
y_lim = np.nanquantile(np.abs(points_df['a_fb']), 0.99)

x_lim = max(0.5, x_lim) * 1.1
y_lim = max(0.5, y_lim) * 1.1

ax.set_xlim(-x_lim, x_lim)
ax.set_ylim(-y_lim, y_lim)

ax.set_xlabel('Left / Right acceleration (m/s²)\nLeft ←        → Right')
ax.set_ylabel('Front / Back acceleration (m/s²)\nDecelerating ↓        ↑ Accelerating')
ax.set_title(f'Top-down acceleration density across {points_df["sprint_num"].nunique()} sprints')

ax.grid(True, alpha=0.3)
plt.show()

# Subsetting Data

In [ ]:
from pathlib import Path
import pandas as pd

# Subsetting youth data to save space
folder = Path("") # enter the folder path here

# Finding files 
for file in folder.glob(""): # enter the naming convention of the files here
    df = pd.read_csv(file)

    # Keeping every 10th row and only the first 5 columns
    df_subset = df.iloc[::10, :5].reset_index(drop=True)

    # Creating new filename ending in _subset.csv
    new_name = file.stem + "_subset.csv"
    output_file = folder / new_name

    # Saving new file
    df_subset.to_csv(output_file, index=False)

    print(f"Saved: {output_file}");

In [ ]:
# Subsetting senior data to save space
folder = Path("") # enter the folder path here 

# Finding files 
for file in folder.glob(""): # enter the naming convention of the files here
    df = pd.read_csv(file)

    # Keeping every 10th row and only the first 5 columns
    df_subset = df.iloc[::10, :5].reset_index(drop=True)

    # Creating new filename ending in _subset.csv
    new_name = file.stem + "_subset.csv"
    output_file = folder / new_name

    # Saving new file
    df_subset.to_csv(output_file, index=False)

    print(f"Saved: {output_file}")

# Reproducible pipeline for exploring acceleration output

## This cell contains all pre-defined functions:

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import scipy.signal
import matplotlib.pyplot as plt

# -----------------------------
# Constants
# -----------------------------
QUADRANT_ORDER = [
    "accelerating_forward_left",
    "accelerating_forward_right",
    "decelerating_left",
    "decelerating_right",
]

REQUIRED_COLS = [
    "Player Display Name",
    "Time",
    "Lat",
    "Lon",
    "Speed (m/s)",
]

# -----------------------------
# Signal helpers
# -----------------------------
def butter_lowpass_array(x, fs, cutoff=2.0, order=4):
    x = np.asarray(x, dtype=float)

    if len(x) < 20:
        return x.copy()

    nyquist = fs / 2.0
    if nyquist <= 0 or cutoff >= nyquist:
        return x.copy()

    Wn = cutoff / nyquist
    b, a = scipy.signal.butter(order, Wn, btype="low")

    padlen = 3 * (max(len(a), len(b)) - 1)
    if len(x) <= padlen:
        return x.copy()

    return scipy.signal.filtfilt(b, a, x)

def wrap_to_pi(angle):
    return (angle + np.pi) % (2 * np.pi) - np.pi

# -----------------------------
# Coordinate helpers
# -----------------------------
def add_local_xy(df_in):
    R = 6371000.0
    out = df_in.copy()

    lat0 = np.deg2rad(out["Lat"].iloc[0])
    lon0 = np.deg2rad(out["Lon"].iloc[0])

    lat = np.deg2rad(out["Lat"].to_numpy())
    lon = np.deg2rad(out["Lon"].to_numpy())

    out["X"] = (lon - lon0) * np.cos(lat0) * R
    out["Y"] = (lat - lat0) * R
    return out

# -----------------------------
# Time helpers
# -----------------------------
def coerce_time_to_seconds(time_series):
    """
    Parse values like 16:43:21.6 as elapsed time and convert to seconds
    relative to the first sample.
    """
    s = pd.Series(time_series).astype(str).str.strip()

    td = pd.to_timedelta(s, errors="coerce")
    if td.notna().sum() >= max(3, int(0.8 * len(s))):
        sec = td.dt.total_seconds()
        return sec - sec.dropna().iloc[0]

    num = pd.to_numeric(s, errors="coerce")
    if num.notna().sum() >= max(3, int(0.8 * len(s))):
        return num - num.dropna().iloc[0]

    raise ValueError("Could not parse Time column into elapsed seconds.")

def estimate_fs(time_s):
    dt = np.diff(np.asarray(time_s, dtype=float))
    dt = dt[np.isfinite(dt) & (dt > 0)]
    if len(dt) == 0:
        raise ValueError("Cannot estimate sampling frequency from Time.")
    return 1.0 / np.median(dt)

# -----------------------------
# File / metadata helpers
# -----------------------------
def parse_match_info(file_path):
    """
    Expected filename patterns:
      Ards_youth_YYYY-MM-DD-Team_on_pitch_subset.csv
      Ards_senior_YYYY-MM-DD-Team_on_pitch_subset.csv

    Opponent is ignored in the match_id.
    """
    stem = Path(file_path).stem

    pattern = re.compile(
        r"^(Ards_(?P<squad>youth|senior))_(?P<date>\d{4}-\d{2}-\d{2})-(?P<opponent>.+?)_on_pitch_subset$",
        flags=re.IGNORECASE
    )
    m = pattern.match(stem)

    if m:
        squad = m.group("squad").lower()
        match_date = m.group("date")
        opponent = m.group("opponent")
        match_id = match_date
    else:
        squad = "unknown"
        match_date = None
        opponent = None
        match_id = stem

    return {
        "source_file": str(file_path),
        "file_name": Path(file_path).name,
        "match_id": match_id,
        "match_date": match_date,
        "opponent": opponent,
        "squad": squad,
    }

def select_match_files(data_dir, match_dates=None, filename_contains=None):
    files = sorted(Path(data_dir).glob("*.csv"))

    if match_dates is not None:
        match_dates = set(match_dates)
        files = [f for f in files if any(f"_{d}-" in f.stem for d in match_dates)]

    if filename_contains is not None:
        if isinstance(filename_contains, str):
            filename_contains = [filename_contains]
        terms = [str(t).lower() for t in filename_contains]
        files = [f for f in files if any(t in f.stem.lower() for t in terms)]

    return files

def list_players_in_folder(data_dir, n_files=None):
    files = sorted(Path(data_dir).glob("*.csv"))
    if n_files is not None:
        files = files[:n_files]

    players = set()
    for f in files:
        try:
            df = pd.read_csv(f)
            df.columns = df.columns.astype(str).str.strip()
            if "Player Display Name" in df.columns:
                vals = df["Player Display Name"].dropna().astype(str).str.strip().unique()
                players.update(vals)
        except Exception:
            pass

    return sorted(players)

def player_exists_in_file(file_path, player_name):
    try:
        df = pd.read_csv(file_path, usecols=["Player Display Name"])
    except Exception:
        df = pd.read_csv(file_path)
        df.columns = df.columns.astype(str).str.strip()
        if "Player Display Name" not in df.columns:
            return False
        df = df[["Player Display Name"]]

    df.columns = df.columns.astype(str).str.strip()
    names = df["Player Display Name"].dropna().astype(str).str.strip()
    return str(player_name).strip() in set(names)

def get_player_match_files(data_dir, player_name):
    """
    Return all CSV files in date order where this player appears.
    """
    files = sorted(Path(data_dir).glob("*.csv"))

    player_files = []
    for f in files:
        try:
            if player_exists_in_file(f, player_name):
                player_files.append(f)
        except Exception:
            pass

    player_files = sorted(
        player_files,
        key=lambda f: (
            parse_match_info(f).get("match_date") is None,
            parse_match_info(f).get("match_date") or "",
            f.name
        )
    )
    return player_files

def select_longitudinal_files(
    player_files,
    n_matches=10,
    method="evenly_spaced",
    every_n=None,
    start_index=0,
    custom_indices=None,
):
    """
    method:
      - evenly_spaced
      - every_nth
      - custom_indices
      - first_n
      - last_n
      - all
    """
    if len(player_files) == 0:
        return []

    if method == "all":
        return list(player_files)

    if n_matches is None or n_matches >= len(player_files):
        return list(player_files)

    if method == "evenly_spaced":
        idx = np.linspace(0, len(player_files) - 1, n_matches, dtype=int)
        idx = np.unique(idx)
        return [player_files[i] for i in idx]

    elif method == "every_nth":
        if every_n is None or every_n <= 0:
            raise ValueError("For method='every_nth', provide every_n > 0.")
        selected = player_files[start_index::every_n]
        return selected[:n_matches] if n_matches is not None else selected

    elif method == "custom_indices":
        if custom_indices is None:
            raise ValueError("For method='custom_indices', provide custom_indices.")
        idx = [i for i in custom_indices if 0 <= i < len(player_files)]
        return [player_files[i] for i in idx]

    elif method == "first_n":
        return player_files[:n_matches]

    elif method == "last_n":
        return player_files[-n_matches:]

    else:
        raise ValueError(f"Unknown method: {method}")

def show_selected_matches(selected_files):
    rows = []
    for i, f in enumerate(selected_files):
        meta = parse_match_info(f)
        rows.append({
            "selection_order": i,
            "file_name": f.name,
            "match_date": meta.get("match_date"),
        })
    return pd.DataFrame(rows)

# -----------------------------
# Sprint logic
# -----------------------------
def find_sprints(player_df, FS, sprint_thr=7.0, jog_thr=2.0, min_dur_s=0.8):
    is_sprint = (player_df["Filtered Speed"] > sprint_thr).to_numpy().astype(int)
    change = np.diff(is_sprint, prepend=is_sprint[0])

    starts = np.where(change == 1)[0].tolist()
    ends = np.where(change == -1)[0].tolist()

    if is_sprint[0] == 1 and (len(starts) == 0 or starts[0] != 0):
        starts = [0] + starts
    if len(ends) < len(starts):
        ends.append(len(player_df) - 1)

    expanded = []
    min_len = int(np.ceil(min_dur_s * FS))

    for s, e in zip(starts, ends):
        i = s
        while i > 0 and player_df.loc[i, "Filtered Speed"] > jog_thr:
            i -= 1
        s_exp = i

        i = e
        while i < len(player_df) - 1 and player_df.loc[i, "Filtered Speed"] > jog_thr:
            i += 1
        e_exp = i

        if (e_exp - s_exp + 1) >= min_len:
            expanded.append((s_exp, e_exp))

    merged = []
    for s, e in sorted(expanded):
        if not merged or s > merged[-1][1]:
            merged.append([s, e])
        else:
            merged[-1][1] = max(merged[-1][1], e)

    return [(s, e) for s, e in merged]

# -----------------------------
# Per-match preparation
# -----------------------------
def prepare_player_match_df(
    file_path,
    player_name,
    fs=None,
    speed_cutoff=2.0,
    xy_cutoff=2.0,
    filter_order=4,
):
    df = pd.read_csv(file_path)
    df.columns = df.columns.astype(str).str.strip()

    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"{file_path} missing required columns: {missing}")

    df["Player Display Name"] = df["Player Display Name"].astype(str).str.strip()
    player_name = str(player_name).strip()

    df = df[df["Player Display Name"] == player_name].copy()
    if df.empty:
        return None, None, parse_match_info(file_path)

    df["time_s"] = coerce_time_to_seconds(df["Time"])

    for col in ["Lat", "Lon", "Speed (m/s)"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = (
        df.dropna(subset=["time_s", "Lat", "Lon", "Speed (m/s)"])
          .sort_values("time_s")
          .drop_duplicates(subset=["time_s"])
          .reset_index(drop=True)
    )

    if len(df) < 5:
        return None, None, parse_match_info(file_path)

    if fs is None:
        fs = estimate_fs(df["time_s"])

    df["Filtered Speed"] = butter_lowpass_array(
        df["Speed (m/s)"], fs=fs, cutoff=speed_cutoff, order=filter_order
    )

    df = add_local_xy(df)
    df["X_filt"] = butter_lowpass_array(df["X"], fs=fs, cutoff=xy_cutoff, order=filter_order)
    df["Y_filt"] = butter_lowpass_array(df["Y"], fs=fs, cutoff=xy_cutoff, order=filter_order)

    return df, fs, parse_match_info(file_path)

# -----------------------------
# Accel-direction cloud
# -----------------------------
def build_accel_direction_cloud(
    player_df,
    sprint_windows,
    min_speed_for_direction=2.0,
    player_mass_kg=None,
):
    records = []

    for sprint_num, (s_idx, e_idx) in enumerate(sprint_windows, start=1):
        seg = player_df.loc[s_idx:e_idx].copy().reset_index(drop=True)

        if len(seg) < 4:
            continue

        t = seg["time_s"].to_numpy(dtype=float)
        x = seg["X_filt"].to_numpy(dtype=float)
        y = seg["Y_filt"].to_numpy(dtype=float)
        s = seg["Filtered Speed"].to_numpy(dtype=float)

        dt = np.diff(t)
        if np.any(dt <= 0):
            continue

        dx = np.diff(x)
        dy = np.diff(y)
        disp = np.hypot(dx, dy)

        valid_motion = disp > 1e-6
        if valid_motion.sum() < 3:
            continue

        heading = np.unwrap(np.arctan2(dy, dx))
        a_fb_interval = np.diff(s) / dt

        dtheta = wrap_to_pi(np.diff(heading))
        dt_c = 0.5 * (dt[:-1] + dt[1:])             # central timestep
        speed_c = s[1:-1]
        a_fb_c = 0.5 * (a_fb_interval[:-1] + a_fb_interval[1:])
        a_lat = speed_c * (dtheta / dt_c)           # + left, - right

        t_c = t[1:-1]
        heading_deg = np.degrees(heading[1:])

        valid = (
            np.isfinite(t_c)
            & np.isfinite(a_fb_c)
            & np.isfinite(a_lat)
            & np.isfinite(speed_c)
            & np.isfinite(dt_c)
            & (dt_c > 0)
            & (speed_c >= min_speed_for_direction)
        )

        for ti, spd, afb, alat, hdg, dt_local in zip(
            t_c[valid],
            speed_c[valid],
            a_fb_c[valid],
            a_lat[valid],
            heading_deg[valid],
            dt_c[valid],
        ):
            # plotting convention: left negative, right positive
            a_lr_plot = -alat

            if (a_lr_plot < 0) and (afb > 0):
                quadrant = "accelerating_forward_left"
            elif (a_lr_plot > 0) and (afb > 0):
                quadrant = "accelerating_forward_right"
            elif (a_lr_plot < 0) and (afb < 0):
                quadrant = "decelerating_left"
            elif (a_lr_plot > 0) and (afb < 0):
                quadrant = "decelerating_right"
            else:
                quadrant = "axis_or_zero"

            # Resultant acceleration magnitude
            a_mag = np.sqrt(afb**2 + alat**2)

            # Optional force / impulse
            if player_mass_kg is not None:
                force_N = player_mass_kg * a_mag
                impulse_Ns = force_N * dt_local
            else:
                force_N = np.nan
                impulse_Ns = np.nan

            records.append({
                "sprint_num": sprint_num,
                "time_s": ti,
                "dt_s": dt_local,
                "speed_m_s": spd,
                "a_fb": afb,
                "a_lat_leftpos": alat,
                "a_lr_plot": a_lr_plot,
                "a_mag": a_mag,
                "force_N": force_N,
                "impulse_Ns": impulse_Ns,
                "heading_deg": hdg,
                "quadrant": quadrant,
            })

    return pd.DataFrame.from_records(records)

# -----------------------------
# Summaries
# -----------------------------
def summarise_quadrants(points_df):
    valid = points_df[points_df["quadrant"] != "axis_or_zero"].copy()

    if valid.empty:
        overall = pd.DataFrame(columns=["quadrant", "count", "proportion"])
        by_match = pd.DataFrame(columns=["match_id"] + QUADRANT_ORDER + ["total"])
        return overall, by_match

    overall = (
        valid["quadrant"]
        .value_counts()
        .reindex(QUADRANT_ORDER, fill_value=0)
        .rename_axis("quadrant")
        .reset_index(name="count")
    )
    overall["proportion"] = overall["count"] / overall["count"].sum()

    by_match = (
        valid.groupby(["match_id", "quadrant"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=QUADRANT_ORDER, fill_value=0)
        .reset_index()
    )

    by_match["total"] = by_match[QUADRANT_ORDER].sum(axis=1)

    for q in QUADRANT_ORDER:
        by_match[f"{q}_pct"] = np.where(
            by_match["total"] > 0,
            by_match[q] / by_match["total"],
            np.nan
        )

    if "match_id" in by_match.columns:
        by_match = by_match.sort_values("match_id").reset_index(drop=True)

    return overall, by_match

# -----------------------------
# Main analysis
# -----------------------------
def analyse_player_across_matches(
    data_dir,
    player_name,
    match_dates=None,
    filename_contains=None,
    selected_files=None,
    fs=None,
    sprint_thr=7.0,
    jog_thr=2.0,
    min_dur_s=0.8,
    min_speed_for_direction=2.0,
    speed_cutoff=2.0,
    xy_cutoff=2.0,
    filter_order=4,
    player_mass_kg=None,
):
    if selected_files is not None:
        files = list(selected_files)
    else:
        files = select_match_files(
            data_dir=data_dir,
            match_dates=match_dates,
            filename_contains=filename_contains,
        )

    if len(files) == 0:
        print("No CSV files matched the selection.")
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    all_points = []
    skipped = []

    for file_path in files:
        try:
            player_df, fs_used, meta = prepare_player_match_df(
                file_path=file_path,
                player_name=player_name,
                fs=fs,
                speed_cutoff=speed_cutoff,
                xy_cutoff=xy_cutoff,
                filter_order=filter_order,
            )

            if player_df is None or player_df.empty:
                skipped.append({
                    "file": str(file_path),
                    "reason": "player_not_found_or_insufficient_data"
                })
                continue

            sprint_windows = find_sprints(
                player_df,
                FS=fs_used,
                sprint_thr=sprint_thr,
                jog_thr=jog_thr,
                min_dur_s=min_dur_s,
            )

            if len(sprint_windows) == 0:
                skipped.append({
                    "file": str(file_path),
                    "reason": "no_sprints_detected"
                })
                continue

            points_df = build_accel_direction_cloud(
                player_df,
                sprint_windows,
                min_speed_for_direction=min_speed_for_direction,
                player_mass_kg=player_mass_kg,
            )

            if points_df.empty:
                skipped.append({
                    "file": str(file_path),
                    "reason": "no_valid_accel_points"
                })
                continue

            for k, v in meta.items():
                points_df[k] = v

            points_df["player_name"] = player_name
            points_df["fs_used"] = fs_used
            points_df["n_sprints_in_match"] = len(sprint_windows)
            points_df["player_mass_kg"] = player_mass_kg

            all_points.append(points_df)

        except Exception as e:
            skipped.append({
                "file": str(file_path),
                "reason": f"error: {e}"
            })

    if len(all_points) == 0:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame(skipped)

    points_all = pd.concat(all_points, ignore_index=True)
    overall_summary, by_match_summary = summarise_quadrants(points_all)

    return points_all, overall_summary, by_match_summary, pd.DataFrame(skipped)

# -----------------------------
# Plotting
# -----------------------------
def plot_topdown_accel_cloud(points_df, title=None):
    valid = points_df[points_df["quadrant"] != "axis_or_zero"].copy()
    if valid.empty:
        print("No valid quadrant points to plot.")
        return

    fig, ax = plt.subplots(figsize=(9, 9))
    ax.scatter(valid["a_lr_plot"], valid["a_fb"], alpha=0.2, s=18)

    ax.axhline(0, linewidth=1)
    ax.axvline(0, linewidth=1)

    x_lim = np.nanquantile(np.abs(valid["a_lr_plot"]), 0.99)
    y_lim = np.nanquantile(np.abs(valid["a_fb"]), 0.99)

    x_lim = max(0.5, x_lim) * 1.1
    y_lim = max(0.5, y_lim) * 1.1

    ax.set_xlim(-x_lim, x_lim)
    ax.set_ylim(-y_lim, y_lim)

    ax.set_xlabel("Left / Right acceleration (m/s²)\nLeft ←        → Right")
    ax.set_ylabel("Front / Back acceleration (m/s²)\nDecelerating ↓        ↑ Accelerating")

    if title is None:
        player = valid["player_name"].iloc[0] if "player_name" in valid.columns else "Player"
        n_matches = valid["match_id"].nunique() if "match_id" in valid.columns else 1
        title = f"{player}: acceleration cloud across {n_matches} matches"
    ax.set_title(title)

    ax.text(-0.60 * x_lim,  0.82 * y_lim, "Accelerating\nforward-left", ha="center", va="center")
    ax.text( 0.60 * x_lim,  0.82 * y_lim, "Accelerating\nforward-right", ha="center", va="center")
    ax.text(-0.60 * x_lim, -0.82 * y_lim, "Decelerating\nleft", ha="center", va="center")
    ax.text( 0.60 * x_lim, -0.82 * y_lim, "Decelerating\nright", ha="center", va="center")

    ax.grid(True, alpha=0.3)
    plt.show()

def plot_quadrant_breakdown_by_match(by_match_summary, as_percent=False):
    if by_match_summary.empty:
        print("No per-match summary to plot.")
        return

    if as_percent:
        cols = [f"{q}_pct" for q in QUADRANT_ORDER]
        plot_df = by_match_summary.set_index("match_id")[cols].copy()
        plot_df.columns = QUADRANT_ORDER
        ylabel = "Proportion"
        title = "Quadrant proportions by match"
    else:
        plot_df = by_match_summary.set_index("match_id")[QUADRANT_ORDER].copy()
        ylabel = "Count"
        title = "Quadrant counts by match"

    ax = plot_df.plot(kind="bar", stacked=True, figsize=(12, 6))
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(title="Quadrant", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

# -----------------------------
# Force helpers
# -----------------------------
def summarise_quadrant_force(points_df):
    valid = points_df[points_df["quadrant"] != "axis_or_zero"].copy()

    if valid.empty:
        return pd.DataFrame()

    summary = (
        valid.groupby("quadrant")
        .agg(
            n_points=("quadrant", "size"),
            mean_a_mag=("a_mag", "mean"),
            median_a_mag=("a_mag", "median"),
            max_a_mag=("a_mag", "max"),
            mean_force_N=("force_N", "mean"),
            median_force_N=("force_N", "median"),
            max_force_N=("force_N", "max"),
            total_force_N=("force_N", "sum"),
            total_impulse_Ns=("impulse_Ns", "sum"),
        )
        .reindex(QUADRANT_ORDER)
        .reset_index()
    )

    summary["point_pct"] = np.where(
        summary["n_points"].sum() > 0,
        summary["n_points"] / summary["n_points"].sum(),
        np.nan
    )

    summary["force_pct"] = np.where(
        summary["total_force_N"].sum() > 0,
        summary["total_force_N"] / summary["total_force_N"].sum(),
        np.nan
    )

    summary["impulse_pct"] = np.where(
        summary["total_impulse_Ns"].sum() > 0,
        summary["total_impulse_Ns"] / summary["total_impulse_Ns"].sum(),
        np.nan
    )

    return summary

def summarise_quadrant_force_by_match(points_df):
    valid = points_df[points_df["quadrant"] != "axis_or_zero"].copy()

    if valid.empty:
        return pd.DataFrame()

    summary = (
        valid.groupby(["match_id", "quadrant"])
        .agg(
            n_points=("quadrant", "size"),
            total_impulse_Ns=("impulse_Ns", "sum"),
            mean_force_N=("force_N", "mean"),
            max_force_N=("force_N", "max"),
        )
        .reset_index()
    )

    # point proportions by match
    points_wide = (
        summary.pivot(index="match_id", columns="quadrant", values="n_points")
        .reindex(columns=QUADRANT_ORDER, fill_value=0)
        .fillna(0)
    )
    points_wide["total_points"] = points_wide.sum(axis=1)
    for q in QUADRANT_ORDER:
        points_wide[f"{q}_point_pct"] = np.where(
            points_wide["total_points"] > 0,
            points_wide[q] / points_wide["total_points"],
            np.nan
        )

    # impulse proportions by match
    impulse_wide = (
        summary.pivot(index="match_id", columns="quadrant", values="total_impulse_Ns")
        .reindex(columns=QUADRANT_ORDER, fill_value=0)
        .fillna(0)
    )
    impulse_wide["total_impulse_Ns"] = impulse_wide.sum(axis=1)
    for q in QUADRANT_ORDER:
        impulse_wide[f"{q}_impulse_pct"] = np.where(
            impulse_wide["total_impulse_Ns"] > 0,
            impulse_wide[q] / impulse_wide["total_impulse_Ns"],
            np.nan
        )

    out = points_wide.reset_index().merge(
        impulse_wide.reset_index(),
        on="match_id",
        suffixes=("_points", "_impulse")
    )

    return out

def plot_quadrant_points_vs_impulse(summary_df):
    if summary_df.empty:
        print("No quadrant force summary to plot.")
        return

    plot_df = (
        summary_df.set_index("quadrant")[["point_pct", "impulse_pct"]]
        .rename(columns={"point_pct": "Point %", "impulse_pct": "Impulse %"})
    )

    ax = plot_df.plot(kind="bar", figsize=(10, 6))
    ax.set_ylabel("Proportion")
    ax.set_title("Quadrant point proportion vs impulse proportion across")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

def plot_quadrant_impulse_by_match(by_match_force_df):
    if by_match_force_df.empty:
        print("No by-match force summary to plot.")
        return

    cols = [f"{q}_impulse_pct" for q in QUADRANT_ORDER]
    plot_df = by_match_force_df.set_index("match_id")[cols].copy()
    plot_df.columns = QUADRANT_ORDER

    ax = plot_df.plot(kind="bar", stacked=True, figsize=(12, 6))
    ax.set_ylabel("Impulse proportion")
    ax.set_title("Quadrant impulse proportion by match")
    ax.legend(title="Quadrant", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

def plot_mean_force_by_quadrant(summary_df):
    if summary_df.empty:
        print("No quadrant force summary to plot.")
        return

    plot_df = summary_df.set_index("quadrant")[["mean_force_N"]]

    ax = plot_df.plot(kind="bar", figsize=(9, 5), legend=False)
    ax.set_ylabel("Mean force estimate (N)")
    ax.set_title("Mean GPS-derived force estimate by quadrant")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

# -----------------------------
# Export helper
# -----------------------------
def export_analysis_outputs(
    output_dir,
    player_name,
    points_all,
    overall_summary,
    by_match_summary,
    skipped,
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    safe_player = re.sub(r"[^A-Za-z0-9_-]+", "_", str(player_name).strip())

    points_path = output_dir / f"{safe_player}_quadrant_points_all_matches.csv"
    overall_path = output_dir / f"{safe_player}_quadrant_summary_overall.csv"
    by_match_path = output_dir / f"{safe_player}_quadrant_summary_by_match.csv"
    skipped_path = output_dir / f"{safe_player}_quadrant_skipped_files.csv"

    points_all.to_csv(points_path, index=False)
    overall_summary.to_csv(overall_path, index=False)
    by_match_summary.to_csv(by_match_path, index=False)
    skipped.to_csv(skipped_path, index=False)

    print("Saved:")
    print(points_path)
    print(overall_path)
    print(by_match_path)
    print(skipped_path)

## This cell contains all parameters that can be changed as needed:

In [ ]:
DATA_DIR = "" # enter the folder name containing the raw data     
PLAYER_NAME = "" # enter a player's name here

SPRINT_THRESHOLD = 7.0
JOG_THRESHOLD = 2.0
MIN_SPRINT_DUR_S = 0.8
MIN_SPEED_FOR_DIRECTION = 2.0
PLAYER_MASS_KG = 64.4

SPEED_CUTOFF = 2.0
XY_CUTOFF = 2.0
FILTER_ORDER = 4
FIXED_FS = None

## This cell will find all the matches for the selected player:

In [ ]:
player_files = get_player_match_files(DATA_DIR, PLAYER_NAME)
print(f"Matches found for the player: {len(player_files)}");
show_selected_matches(player_files).head(20);

## The next few cells contain multiple ways to select which matches are to be analysed: 
Choose one of these cells to run whichever method you want to use

In [ ]:
# n evenly spaced matches 
selected_files = select_longitudinal_files(
    player_files,
    n_matches=10,
    method="evenly_spaced"
)

show_selected_matches(selected_files);

In [ ]:
# every nth match
selected_files = select_longitudinal_files(
    player_files,
    n_matches=10,
    method="every_nth",
    every_n=4,
    start_index=0
)

show_selected_matches(selected_files);

In [ ]:
# custom indices
selected_files = select_longitudinal_files(
    player_files,
    method="custom_indices",
    custom_indices=[70]
)

show_selected_matches(selected_files);

In [ ]:
# first n matches 
selected_files = select_longitudinal_files(
    player_files,
    n_matches=10,
    method="first_n"
)

show_selected_matches(selected_files);

In [ ]:
# last n matches
selected_files = select_longitudinal_files(
    player_files,
    n_matches=10,
    method="last_n"
)

show_selected_matches(selected_files);

## This cell will run the analysis:

In [ ]:
points_all, overall_summary, by_match_summary, skipped = analyse_player_across_matches(
    data_dir=DATA_DIR,
    player_name=PLAYER_NAME,
    selected_files=selected_files,
    fs=FIXED_FS,
    sprint_thr=SPRINT_THRESHOLD,
    jog_thr=JOG_THRESHOLD,
    min_dur_s=MIN_SPRINT_DUR_S,
    min_speed_for_direction=MIN_SPEED_FOR_DIRECTION,
    speed_cutoff=SPEED_CUTOFF,
    xy_cutoff=XY_CUTOFF,
    filter_order=FILTER_ORDER,
    player_mass_kg=PLAYER_MASS_KG,
)

print(f"Total valid points: {len(points_all)}")
print(f"Matches included: {points_all['match_id'].nunique() if not points_all.empty else 0}")
print(f"Files skipped: {len(skipped)}")

## These cells below will produce multiple summaries to inspect:

In [ ]:
overall_summary

In [ ]:
by_match_summary

In [ ]:
points_all.head();

## These cells below will produce multiple visualisations to inspect:

In [ ]:
plot_topdown_accel_cloud(points_all, title=f"{PLAYER_NAME} quadrant cloud")
plot_quadrant_breakdown_by_match(by_match_summary, as_percent=False)
plot_quadrant_breakdown_by_match(by_match_summary, as_percent=True)

## This cell will export the results if needed:

In [ ]:
export_analysis_outputs(
    output_dir="analysis_outputs",
    player_name=PLAYER_NAME,
    points_all=points_all,
    overall_summary=overall_summary,
    by_match_summary=by_match_summary,
    skipped=skipped,
)

## Force Analysis:

In [ ]:
quadrant_force_summary = summarise_quadrant_force(points_all)
quadrant_force_summary

In [ ]:
quadrant_force_by_match = summarise_quadrant_force_by_match(points_all)
quadrant_force_by_match.head()

In [ ]:
plot_quadrant_points_vs_impulse(quadrant_force_summary)
plot_mean_force_by_quadrant(quadrant_force_summary)
plot_quadrant_impulse_by_match(quadrant_force_by_match)